# Module 5: mT5 and ByT5 Experiments

Преподаватель предложил протестировать модели `mT5-small`, `mT5-base` и `ByT5-small` (архитектура T5). 
Важно понимать ключевое отличие: **ruGPT-3** — это авторегрессионная модель (decoder-only), которая изначально предназначена для генерации текста (Causal LM). 
**mT5** и **ByT5** — это модели кодер-декодер (Seq2Seq). Чтобы научить их генерировать текст в определенном стиле (продолжать мысль), нам нужно подавать начало предложения (prompt/prefix) в энкодер, а ожидаемое продолжение — в декодер.

Здесь мы подготовим пайплайн под Seq2Seq-обучение.

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device in use: {device}")

In [ ]:
# Загружаем данные
df_train = pd.read_csv('data/train_sentences.csv')

# Для Seq2Seq подхода нам нужно разделить предложения на "вход" (prefix/prompt) и "выход" (target/continuation).
# Простой подход: берем первые 2-3 слова как input, а остаток как target.
def split_sentence(sentence):
    words = sentence.split()
    if len(words) > 3:
        return " ".join(words[:2]), " ".join(words[2:])
    return sentence, sentence

df_train['prefix'], df_train['target'] = zip(*df_train['sentence'].apply(split_sentence))

dataset = Dataset.from_pandas(df_train)
print(f"Dataset loaded. Number of sentences: {len(dataset)}")
print("Example:")
print(f"Prefix: {dataset[0]['prefix']}")
print(f"Target: {dataset[0]['target']}")

In [ ]:
# Выбор модели. Раскомментируйте нужную:
# model_name = 'google/mt5-small'
# model_name = 'google/mt5-base'
model_name = 'google/mt5-small'

print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loading model {model_name}...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)

# Для ByT5 (байт-левел) sequence length нужно ставить больше, так как 1 символ = 1-4 токена (байта).
# Для mT5 достаточно 128.
max_length = 256 if 'byt5' in model_name else 64
print(f"Using max_length={max_length}")

In [ ]:
def tokenize_function(examples):
    # Токенизируем вход (начало предложения)
    model_inputs = tokenizer(
        examples['prefix'],
        max_length=max_length,
        truncation=True,
        padding='max_length'
    )
    
    # Токенизируем выход (продолжение предложения)
    labels = tokenizer(
        text_target=examples['target'],
        max_length=max_length,
        truncation=True,
        padding='max_length'
    )
    
    # Заменяем pad_token_id в labels на -100, чтобы они игнорировались при подсчете Loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['sentence', 'prefix', 'target']
)
tokenized_dataset.set_format('torch')
print("Done!")

In [ ]:
model_safename = model_name.replace("/", "-")
training_args = Seq2SeqTrainingArguments(
    output_dir=f'./results_{model_safename}',
    overwrite_output_dir=True,
    num_train_epochs=1, # можно увеличить до 5-10 для t5
    per_device_train_batch_size=8,
    save_steps=200,
    save_total_limit=2,
    logging_steps=25,
    warmup_steps=50,
    weight_decay=0.01,
    optim='adamw_torch',
    fp16=(device == 'cuda' and 'byt5' not in model_name), # ByT5 иногда нестабилен в fp16
    predict_with_generate=True,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Starting fine-tuning for {model_name}...")
trainer.train()
print("Training completed!")

In [ ]:
save_path = f'./abai_{model_safename}_final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

In [ ]:
from transformers import pipeline

generator = pipeline(
    'text2text-generation', # Для T5 используем text2text, а не text-generation
    model=save_path,
    tokenizer=save_path,
    device=0 if device == 'cuda' else -1
)

prompt = "Адам баласы"
output = generator(
    prompt,
    max_new_tokens=80,
    num_return_sequences=3,
    temperature=0.9,
    do_sample=True,
    top_k=50,
    top_p=0.92,
)

print(f"\n=== Text Generation (Prompt: '{prompt}') ===\n")
for i, result in enumerate(output):
    print(f"--- Option {i+1} ---")
    print(prompt + " " + result['generated_text'])
    print()